# Homework #8
Jacob A. Fericy

This notebook extends from the notebook we created for Homework 7 using the same UCI Wine Quality data and the same overall engineering style.

Our new tasks is to add **tree-based models** for both the regression and classification tasks, compare those tree-based models against the Homework 7 models on the test set, and then discuss the results.

The two homework tasks are:

1. **Regression** with **alcohol** as the response.
2. **Classification** with **wine type** (red vs. white) as the response.

I keep the code structure consistent with Homework 7:
- dataset
- train/test split
- feature set
- metric definitions
- pipeline-based model organization where appropriate

In [34]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LinearRegression,
    LogisticRegression,
    Ridge,
)
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

random = 37
pd.set_option("display.max_columns", 100)

## Read/Combine the data

I read in the two UCI wine files and combined them into one dataframe.  
I used local-file fallbacks first so the notebook can run cleanly if the CSVs are saved in the repo. Otherwise, it can still pull from the UCI URLs. This is a method I used in HW 7, because I was having file issues. 

In [35]:
RED_LOCAL = "winequality-red.csv"
WHITE_LOCAL = "winequality-white.csv"
RED_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
WHITE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

try:
    red = pd.read_csv(RED_LOCAL, sep=";")
    white = pd.read_csv(WHITE_LOCAL, sep=";")
except FileNotFoundError:
    red = pd.read_csv(RED_URL, sep=";")
    white = pd.read_csv(WHITE_URL, sep=";")

red["wine_type"] = "red"
white["wine_type"] = "white"

wine = pd.concat([red, white], ignore_index=True)
wine["wine_type_binary"] = (wine["wine_type"] == "white").astype(int)

wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type,wine_type_binary
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,0


In [36]:
wine.shape, wine["wine_type"].value_counts()

((6497, 14),
 wine_type
 white    4898
 red      1599
 Name: count, dtype: int64)

## Train/test split

As we used in Homework 7, I use a **stratified split** so the training and test sets preserve the same red/white mix.

In [37]:
feature_cols = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
    "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "quality"
]

train_df, test_df = train_test_split(
    wine,
    test_size=0.20,
    stratify=wine["wine_type"],
    random_state=random ,
)

train_df["wine_type"].value_counts(normalize=True), test_df["wine_type"].value_counts(normalize=True)

(wine_type
 white    0.753896
 red      0.246104
 Name: proportion, dtype: float64,
 wine_type
 white    0.753846
 red      0.246154
 Name: proportion, dtype: float64)

## Regression task: alcohol as the response

I also kept the Homework 7 regression models so I can compare them directly against the new tree-based models.

The Homework 7 regression candidates are:
1. **All-linear model**
2. **Selected linear model**
3. **Interaction model**
4. **Polynomial model**

I then carry forward:
- Best MLR-style model from cross-validation
- LASSO
- Ridge
- Elastic Net

For Homework 8, I add:
- a **regression tree**
- a **random forest regressor**

The final comparison on the test set uses:
- **RMSE**
- **MAE**

In [38]:
X_train_reg = train_df[feature_cols]
y_train_reg = train_df["alcohol"]
X_test_reg = test_df[feature_cols]
y_test_reg = test_df["alcohol"]

cv = KFold(n_splits=5, shuffle=True, random_state = random)

selected_cols = ["volatile acidity", "chlorides", "total sulfur dioxide", "density", "sulphates", "quality"]
interaction_cols = ["fixed acidity", "volatile acidity", "citric acid", "density", "sulphates"]
poly_cols = ["fixed acidity", "volatile acidity", "citric acid", "density", "quality"]

mlr_candidates = {
    "MLR_All_Linear": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "MLR_Selected_Linear": Pipeline([
        ("prep", ColumnTransformer([
            ("selected", StandardScaler(), selected_cols),
        ], remainder="drop")),
        ("model", LinearRegression()),
    ]),
    "MLR_Interactions": Pipeline([
        ("prep", ColumnTransformer([
            (
                "interactions",
                Pipeline([
                    ("scale", StandardScaler()),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
                ]),
                interaction_cols,
            )
        ], remainder="drop")),
        ("model", LinearRegression()),
    ]),
    "MLR_Polynomial": Pipeline([
        ("prep", ColumnTransformer([
            (
                "poly",
                Pipeline([
                    ("scale", StandardScaler()),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ]),
                poly_cols,
            )
        ], remainder="drop")),
        ("model", LinearRegression()),
    ]),
}

def cv_regression_summary(models, X, y, cv):
    rows = []
    for name, model in models.items():
        rmse_grid = GridSearchCV(
            estimator=model,
            param_grid={},
            scoring="neg_mean_squared_error",
            cv=cv,
            refit=True,
            n_jobs=-1,
        )
        mae_grid = GridSearchCV(
            estimator=model,
            param_grid={},
            scoring="neg_mean_absolute_error",
            cv=cv,
            refit=True,
            n_jobs=-1,
        )

        rmse_grid.fit(X, y)
        mae_grid.fit(X, y)

        rows.append({
            "model": name,
            "cv_rmse": np.sqrt(-rmse_grid.best_score_),
            "cv_mae": -mae_grid.best_score_,
        })

    return pd.DataFrame(rows).sort_values(["cv_rmse", "cv_mae"]).reset_index(drop=True)

mlr_cv_results = cv_regression_summary(mlr_candidates, X_train_reg, y_train_reg, cv)
mlr_cv_results

,model,cv_rmse,cv_mae
0,MLR_All_Linear,0.555310,0.399661
1,MLR_Polynomial,0.711836,0.528873
2,MLR_Selected_Linear,0.747594,0.578205
3,MLR_Interactions,0.844889,0.618683


In [39]:
best_mlr_name = mlr_cv_results.loc[0, "model"]

ridge_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(solver="svd"))
])

ridge_grid = {
    "model__alpha": np.logspace(-3, 3, 60)
}

ridge_cv = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

ridge_cv.fit(X_train_reg, y_train_reg)

base_regression_models = {
    f"Best_{best_mlr_name}": clone(mlr_candidates[best_mlr_name]).fit(X_train_reg, y_train_reg),
    "LASSO": Pipeline([
        ("scale", StandardScaler()),
        ("model", LassoCV(
            alphas=np.logspace(-3, 1, 50),
            cv=cv,
            max_iter=20000,
            random_state=random,
        )),
    ]).fit(X_train_reg, y_train_reg),
    "Ridge": ridge_cv.best_estimator_,
    "ElasticNet": Pipeline([
        ("scale", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-3, 1, 40),
            cv=cv,
            max_iter=20000,
            random_state=random,
        )),
    ]).fit(X_train_reg, y_train_reg),
}

base_regression_models.keys()

dict_keys(['Best_MLR_All_Linear', 'LASSO', 'Ridge', 'ElasticNet'])

### Tree-based regression models

The regression tree may capture nonlinear splits and interactions automatically.  
The random forest improves on a single tree by averaging across many trees fit to bootstrap samples, which usually reduces variance and improves out-of-sample performance.

I tune:
- **Regression tree:** `max_depth`, `min_samples_leaf`
- **Random forest:** `max_features`, `min_samples_leaf`

I have use RMSE during training-time tuning.

In [40]:
reg_tree_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state = random),
    param_grid={
        "max_depth": [3, 5, 8, None],
        "min_samples_leaf": [1, 5, 10],
    },
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

reg_tree_grid.fit(X_train_reg, y_train_reg)

reg_forest_grid = GridSearchCV(
    estimator=RandomForestRegressor(
        n_estimators=150,
        random_state = random,
        n_jobs=-1,
    ),
    param_grid={
        "max_features": [3, 5, None],
        "min_samples_leaf": [1, 5],
    },
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

reg_forest_grid.fit(X_train_reg, y_train_reg)

reg_tree_grid.best_params_, reg_forest_grid.best_params_

({'max_depth': None, 'min_samples_leaf': 10},
 {'max_features': None, 'min_samples_leaf': 1})

In [41]:
all_regression_models = dict(base_regression_models)
all_regression_models["RegressionTree"] = reg_tree_grid.best_estimator_
all_regression_models["RandomForest"] = reg_forest_grid.best_estimator_

def regression_test_summary(models, X_test, y_test):
    rows = []
    for name, model in models.items():
        pred = model.predict(X_test)
        rows.append({
            "model": name,
            "test_rmse": mean_squared_error(y_test, pred, squared=False),
            "test_mae": mean_absolute_error(y_test, pred),
        })
    return pd.DataFrame(rows).sort_values(["test_rmse", "test_mae"]).reset_index(drop=True)

regression_test_results = regression_test_summary(all_regression_models, X_test_reg, y_test_reg)
regression_test_results

,model,test_rmse,test_mae
0,RandomForest,0.390120,0.259449
1,Best_MLR_All_Linear,0.477681,0.375331
2,LASSO,0.477884,0.375648
3,Ridge,0.477917,0.375580
4,ElasticNet,0.477935,0.375692
5,RegressionTree,0.543255,0.389480


## Classification task: wine type as the response

With Homework 7, I used the wine type indicator as the binary response.  
I keep the Homework 7 logistic models and then add:
- a **classification tree**
- a **random forest classifier**

For model selection during training, the homework asks to use:
- **log-loss** or **negative log-loss**

For final test-set comparison, I report:
- **log-loss**
- **accuracy**

In [42]:
X_train_clf = train_df[feature_cols]
y_train_clf = train_df["wine_type_binary"]
X_test_clf = test_df[feature_cols]
y_test_clf = test_df["wine_type_binary"]

logit_candidates = {
    "Logit_All_Linear": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
    ]),
    "Logit_Selected_Linear": Pipeline([
        ("prep", ColumnTransformer([
            ("selected", StandardScaler(), selected_cols),
        ], remainder="drop")),
        ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
    ]),
    "Logit_Interactions": Pipeline([
        ("prep", ColumnTransformer([
            (
                "interactions",
                Pipeline([
                    ("scale", StandardScaler()),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
                ]),
                interaction_cols,
            )
        ], remainder="drop")),
        ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
    ]),
    "Logit_Polynomial": Pipeline([
        ("prep", ColumnTransformer([
            (
                "poly",
                Pipeline([
                    ("scale", StandardScaler()),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ]),
                poly_cols,
            )
        ], remainder="drop")),
        ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
    ]),
}

def classification_cv_summary(models, X, y, cv):
    rows = []
    for name, model in models.items():
        grid = GridSearchCV(
            estimator=model,
            param_grid={},
            scoring="neg_log_loss",
            cv=cv,
            refit=True,
            n_jobs=-1,
        )
        grid.fit(X, y)
        rows.append({
            "model": name,
            "cv_log_loss": -grid.best_score_,
        })
    return pd.DataFrame(rows).sort_values("cv_log_loss").reset_index(drop=True)

classification_cv_results = classification_cv_summary(logit_candidates, X_train_clf, y_train_clf, cv)
classification_cv_results

,model,cv_log_loss
0,Logit_All_Linear,0.044820
1,Logit_Selected_Linear,0.061238
2,Logit_Interactions,0.171188
3,Logit_Polynomial,0.176906


### Tree-based classification models

The classification tree creates recursive splits to separate red and white wines.  
The random forest classifier averages across many classification trees and usually improves stability and predictive performance.

I tune:
- **Classification tree:** `max_depth`, `min_samples_leaf`
- **Random forest classifier:** `max_features`, `min_samples_leaf`

I used **negative log-loss** for the tuning stage.

In [43]:
clf_tree_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state = random),
    param_grid={
        "max_depth": [3, 5, 8, None],
        "min_samples_leaf": [1, 5, 10],
    },
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=-1,
)

clf_tree_grid.fit(X_train_clf, y_train_clf)

clf_forest_grid = GridSearchCV(
    estimator=RandomForestClassifier(
        n_estimators=150,
        random_state=random,
        n_jobs=-1,
    ),
    param_grid={
        "max_features": [3, 5, None],
        "min_samples_leaf": [1, 5],
    },
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=-1,
)

clf_forest_grid.fit(X_train_clf, y_train_clf)

clf_tree_grid.best_params_, clf_forest_grid.best_params_

({'max_depth': 3, 'min_samples_leaf': 10},
 {'max_features': 3, 'min_samples_leaf': 1})

In [44]:
fit_logit_models = {
    name: clone(model).fit(X_train_clf, y_train_clf)
    for name, model in logit_candidates.items()
}

all_classification_models = dict(fit_logit_models)
all_classification_models["ClassificationTree"] = clf_tree_grid.best_estimator_
all_classification_models["RandomForest"] = clf_forest_grid.best_estimator_

def classification_test_summary(models, X_test, y_test):
    rows = []
    for name, model in models.items():
        prob = model.predict_proba(X_test)
        pred = model.predict(X_test)
        rows.append({
            "model": name,
            "test_log_loss": log_loss(y_test, prob),
            "test_accuracy": accuracy_score(y_test, pred),
        })
    return pd.DataFrame(rows).sort_values(
        ["test_log_loss", "test_accuracy"],
        ascending=[True, False]
    ).reset_index(drop=True)

classification_test_results = classification_test_summary(
    all_classification_models,
    X_test_clf,
    y_test_clf,
)
classification_test_results

,model,test_log_loss,test_accuracy
0,RandomForest,0.019763,0.997692
1,Logit_All_Linear,0.027026,0.993077
2,Logit_Selected_Linear,0.042138,0.986154
3,ClassificationTree,0.091076,0.969231
4,Logit_Interactions,0.137795,0.953846
5,Logit_Polynomial,0.146750,0.941538


## Final results

In this assignment, we added tree-based models to extend our modeling away from linear based data to a more real world example and used data to show this. The advantage is also around the method and how they read in the data. Tree based methods learn their struture in an automated fashion. Random Forests also can reduce overfitting.

For the **regression** task, I compare the Homework 7 regression models against the new tree-based regression models using **RMSE** and **MAE** on the test set.

For the **classification** task, I compare the Homework 7 logistic models against the new tree-based classification models using **log-loss** and **accuracy** on the test set.

Based on the results, Logit_All_Linear in my model is best, given the smaller log_loss. RandomForest has the high test accuracy, but log-loss is a better given predicted probabilities are more accurate and log-loss penalizes probability estimates, which can be more informative when we are dealing with less than robust probability inputs. 

